# RAFNet embedding extraction (Kaggle GPU)
1. Upload `data/manifest.csv` + a zip of your audio to Kaggle input, or mount your dataset.
2. Run all cells (needs a GPU accelerator: Settings → Accelerator → GPU T4).
3. Download `embed_cache/` output and place it in the repo `data/embed_cache/`.
Windows of 4 s use the SAME preprocessing as inference (`preprocess.fixed_window`).

In [ ]:
!pip install -q torch torchaudio transformers soundfile pandas scikit-learn
import torch
print(torch.__version__, torch.cuda.is_available())

In [ ]:
import csv, numpy as np, soundfile as sf, torch
from pathlib import Path
from transformers import WavLMModel, WavLMConfig

MANIFEST = '/kaggle/input/rafnet-data/manifest.csv'  # <-- adjust to your input
OUT = Path('/kaggle/working/embed_cache'); OUT.mkdir(parents=True, exist_ok=True)
SR, WIN = 16000, 4.0  # identical contract to soc/preprocess.py

def to_model_input(path):
    x, sr = sf.read(path, dtype='float32', always_2d=False)
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x.mean(axis=1)
    if sr != SR:  # genuine linear resample, same as inference
        t_old = np.linspace(0, 1, len(x)); t_new = np.linspace(0, 1, max(1, int(round(len(x)/sr*SR))))
        x = np.interp(t_new, t_old, x).astype(np.float32)
    want = int(WIN * SR)
    x = x[:want] if len(x) >= want else np.pad(x, (0, want - len(x)))
    return np.clip(x, -1, 1)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
enc = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(device).eval()
print('encoder on', device)

In [ ]:
rows = list(csv.DictReader(open(MANIFEST, encoding='utf-8')))
print(len(rows), 'manifest rows')
shard, buf, SHARD_N = 0, [], 512
@torch.inference_mode()
def embed(wav):
    t = torch.from_numpy(wav).unsqueeze(0).to(device)
    h = enc(t).last_hidden_state  # [1, T, 768]
    return h.mean(dim=1).squeeze(0).cpu().numpy().astype('float32')
for i, r in enumerate(rows):
    try:
        buf.append({'x': embed(to_model_input(r['path'])), **{k: r[k] for k in ('label_origin','generator_family','split')}})
    except Exception as e:
        print('skip', r['path'], str(e)[:120])
    if len(buf) >= SHARD_N or i == len(rows) - 1:
        torch.save(buf, OUT / f'shard_{shard:03d}.pt'); buf = []; shard += 1
        print('saved shard', shard)

Download `/kaggle/working/embed_cache/*.pt` → repo `data/embed_cache/`, then run `notebooks/train_colab.ipynb` (or the local head trainer on CPU).